# **HAIO Országos Döntő 2024**

**Versenyző neve:**

# Transzfer-tanulás

A modern mesterséges neurális hálók felé támasztott egyik elvárásunk, hogy képesek legyenek transzferálni az általuk már egy területen/feladaton elsajátított "tudást" más feladatok megoldására is. Például ha a hálónknak megtanítottuk már a kutyák és macskák megkülönböztetését, elvárhatjuk, hogy gyorsabban és kevesebb adatból is meg tudja már tanulni például a lovak és tigrisek megkülönböztetését. Ez számítási erőforrást és időt takarítana meg számunkra, kevés adaton is lehetővé tenné a tanítást, és  potenciálisan javíthatná a modellek teljesítményét a különböző feladatokon. Sajnos általánosan érvényes, garantáltan működő módszer a tudás-transzfer biztosítására nincs egyelőre, a transzfer-tanulás sikeressége adat- és modell-függő a legtöbbször és a különböző rendelkezésre álló technikák még nem teljesen megértettek. Ezekben a feladatokban azt fogjuk vizsgálni, hogy hogyan és mikor, mennyire sikeresen működik a transzfer tanulás.


# Feladatok


**1. Feladat - Transzfer tanítás előtt (x pont)**

Töltsd be két ResNet-20 modellt: az egyik legyen a CIFAR-100 adathalmaz osztályozására betanítva, a másik pedig a CIFAR-10 osztályozására. Írd le, honnan töltöd be a modelleket, illletve, hogy milyen pontosságot érnek el a tesz-halmazokon. Cseréld ki a CIFAR-100 adaton tanított modell utolsó rétegét a CIFAR-10 adaton tanított modellével. Számold ki az így kapott háló pontosságát a CIFAR-10 tanító és teszt adathalmazán külön-külön, és hasonlítsd össze ezeket egy véletlenszerűen inicializált háló (amelynek a kimeneti rétegét szintén lecseréled a CIFAR-10-en tanított modellével) teljesítményével (tehát összesen 4 számot kell megadni), hogy kiderüljön, mennyire történik meg a tudás-transzfer továbbtanítás nélkül a CIFAR-100-ról.  

**2. Feladat - Transzfer tanítás (x pont)**

Csakis az utolsó kimeneti réteg lecserélésével (megfelelő kimeneti neuront tartalmazó, de véletlenszerű paraméterekkel inicializált rétegre kell cserélni), és a többi réteg befagyasztásával tanítsd tovább a CIFAR-10 tanító halmazán a CIFAR-100-on betanított hálót, alkalmasan megválasztott paraméterekkel. Ábrázold egy ábrán a továbbtanítási epochok függvényében, hogyan alakul a háló pontossága a CIFAR-10, illetve a CIFAR-100 teszt-halmazán.

**3. Feladat - Modellek távolsága (x pont)**

Számold ki 3 modell paramétereinek a páronkénti $L_2$-távolságát: a három összehasonlított modell legyen a véletlenszerűen inicializiált, az 1. feladatban CIFAR-100-on tanított modell a lecserélt kiemneti réteggel és a 2. feladatban a CIFAR-10-en továbbtanított modell.

**4. Feladat - Több réteg továbbtanítása (x pont)**
Csak az utolsó réteg tanítása helyett itt már megengedjük, hogy a háló alsóbb rétegei is tanuljanak: a bemenethez közeli rétegeket  az második reziduális blokkal bezárólag is továbbtanítva, lehet-e jobb pontosságot elérni a CIFAR-10 tesz-halmazán? Mi van akkor, ha a csak a második reziduális blokk utáni rétegek tanítása megengedett?

**5. Feladat - Kevés adaton való tanítás (x pont)**

Ha a CIFAR-10 tanítóadatainak csak az 50%-ához férsz hozzá, el tudsz-e érni ugyanolyan jó pontosságot a CIFAR-100-en tanított modell transzfertanításával ezen a kevesebb adaton, mint a teljes CIFAR-10-en tanított modell? Ha sikerül, írd le milyen beállításokkal. Az adatok 20%-ával is sikerül ugyanez?



---
# Előkészületek


## Könyvtárak betöltése, beállítások

In [ ]:
import os
import gc
import csv
import glob
import torch
import multiprocessing

import numpy as np
import pandas as pd
import torch.nn as nn
import matplotlib.pyplot as plt

import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from torch.autograd import Variable

import torchvision
import torchvision.transforms as transforms

A reprodukálható eredmények érdekében elengedhetetlen, hogy rögzítsük a véletlenszám generátor seed-jét.

In [ ]:
# for DL its critical to set the random seed so that students can have a
# baseline to compare their results to expected results.
# Read more here: https://pytorch.org/docs/stable/notes/randomness.html

# Call `set_seed` function in the exercises to ensure reproducibility.

import random
import torch

def set_seed(seed=None, seed_torch=True):
    if seed is None:
        seed = np.random.choice(2 ** 32)
    random.seed(seed)
    np.random.seed(seed)
    if seed_torch:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

    print(f'Random seed {seed} has been set.')

# In case that `DataLoader` is used
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# inform the user if the notebook uses GPU or CPU.

def set_device():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device != "cuda":
        print("WARNING: For this notebook to perform best, "
              "if possible, in the menu under `Runtime` -> "
              "`Change runtime type.`  select `GPU` ")
    else:
        print("GPU is enabled in this notebook.")

    return device

In [ ]:
set_seed(seed=2024)
device = set_device()

Random seed 2024 has been set.


## Hiperparaméterek beállítása



In [ ]:
# hyper-parameters
use_cuda = torch.cuda.is_available()
batch_size = 256
torchvision_transforms = False

---
## A CIFAR-10 és CIFAR-100 adathalmaz betöltése

A CIFAR-100 színes képet tartalmazó adathalmaz 50,000 tanító és 10,000 teszt képpel, ezek 32 x 32 pixelből állnak, és összesen 100 különböző kategóriából lehetnek, mindegyik rendelkezik címkével. A `torchvision.datasets.cifar.CIFAR` objetumon keresztül érhető el az adathalmaz.

In [ ]:
print('==> Preparing data..')

def percentageSplit(full_dataset, percent = 0.0):
    set1_size = int(percent * len(full_dataset))
    set2_size = len(full_dataset) - set1_size
    final_dataset, _ = torch.utils.data.random_split(full_dataset, [set1_size, set2_size])
    return final_dataset


# CIFAR100 normalizing
mean = [0.5071, 0.4866, 0.4409]
std = [0.2673, 0.2564, 0.2762]

# CIFAR10 normalizing
# mean = (0.4914, 0.4822, 0.4465)
# std = (0.2023, 0.1994, 0.2010)

# torchvision transforms
transform_train = transforms.Compose([])
if torchvision_transforms:
    transform_train.transforms.append(transforms.RandomCrop(32, padding=4))
    transform_train.transforms.append(transforms.RandomHorizontalFlip())

transform_train.transforms.append(transforms.ToTensor())
transform_train.transforms.append(transforms.Normalize(mean, std))

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

cifar100_trainset = torchvision.datasets.CIFAR100(
    root='./CIFAR100', train=True, download=True, transform=transform_train)

cifar100_testset = torchvision.datasets.CIFAR100(
    root='./CIFAR100', train=False, download=True, transform=transform_test)

print(f"Objektum: {type(cifar100_trainset)}")
print(f"Tanító adatok shape-je: {cifar100_trainset.data.shape}")
print(f"Teszt adatok shape-je: {cifar100_testset.data.shape}")
print(f"Az osztályok száma: {np.unique(cifar100_trainset.targets).shape[0]}")

num_workers = multiprocessing.cpu_count()

ciafr100_trainloader = torch.utils.data.DataLoader(
    cifar100_trainset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
cifar100_testloader = torch.utils.data.DataLoader(
    cifar100_testset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

==> Preparing data..


100%|██████████| 169001437/169001437 [00:02<00:00, 67302258.38it/s]


Extracting ./CIFAR100/cifar-100-python.tar.gz to ./CIFAR100
Files already downloaded and verified
Objektum: <class 'torchvision.datasets.cifar.CIFAR100'>
Tanító adatok shape-je: (50000, 32, 32, 3)
Teszt adatok shape-je: (10000, 32, 32, 3)
Az osztályok száma: 100


In [ ]:
# CIFAR10 normalizing
mean = (0.4914, 0.4822, 0.4465)
std = (0.2023, 0.1994, 0.2010)

# torchvision transforms
transform_train = transforms.Compose([])
if torchvision_transforms:
    transform_train.transforms.append(transforms.RandomCrop(32, padding=4))
    transform_train.transforms.append(transforms.RandomHorizontalFlip())

transform_train.transforms.append(transforms.ToTensor())
transform_train.transforms.append(transforms.Normalize(mean, std))

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

cifar10_trainset = torchvision.datasets.CIFAR10(
    root='./CIFAR10', train=True, download=True, transform=transform_train)

cifar10_testset = torchvision.datasets.CIFAR10(
    root='./CIFAR10', train=False, download=True, transform=transform_test)

print(f"Objektum: {type(cifar10_trainset)}")
print(f"Tanító adatok shape-je: {cifar10_trainset.data.shape}")
print(f"Teszt adatok shape-je: {cifar10_testset.data.shape}")
print(f"Az osztályok száma: {np.unique(cifar10_trainset.targets).shape[0]}")

num_workers = multiprocessing.cpu_count()

ciafr10_trainloader = torch.utils.data.DataLoader(
    cifar10_trainset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
cifar10_testloader = torch.utils.data.DataLoader(
    cifar10_testset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

100%|██████████| 170498071/170498071 [00:02<00:00, 71657718.43it/s]


Extracting ./CIFAR10/cifar-10-python.tar.gz to ./CIFAR10
Files already downloaded and verified
Objektum: <class 'torchvision.datasets.cifar.CIFAR10'>
Tanító adatok shape-je: (50000, 32, 32, 3)
Teszt adatok shape-je: (10000, 32, 32, 3)
Az osztályok száma: 10


## ResNet modell létrehozása

A ResNet hálóarchitektúra igen népszerű és jó eredményeket elérő modellcsalád alapja, fő jellemzője, hogy reziduális blokkokból áll, a modellcsaládban különböző mélységű modelleket lehet konstruálni a blokkok számának növelésével, pl. ResNet-18, ResNet-32, ResNet-50. A reziduális blokkokon belül jól ismert rétegtípusok, konvolúciós, pooling, bacth normalizáló rétegek találhatóak, és a blokk bemenetét a kimenetéhez adva egy speciális úgynevezett _shortcut_ vagy  _skip connection_ is van. Az [eredeti cikkben](https://arxiv.org/abs/1512.03385) további részleteket olvashattok. Az alábbi kód létrehozza a modell architektúrát, és be is tölt a torchhub-ról [két betanított modellt, amelyekről minden információt itt találtok](https://github.com/chenyaofo/pytorch-cifar-models/).

In [ ]:
# @title ResNet modell architektúra

import torch.nn as nn

class BasicBlock(nn.Module):
  """ResNet in PyTorch.
      Reference:
      [1] Kaiming He, Xiangyu Zhang, Shaoqing Ren, Jian Sun
        Deep Residual Learning for Image Recognition. arXiv:1512.03385
  """

  expansion = 1

  def __init__(self, in_planes, planes, stride=1):
    super(BasicBlock, self).__init__()
    self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
    self.bn1 = nn.BatchNorm2d(planes)
    self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
    self.bn2 = nn.BatchNorm2d(planes)

    self.shortcut = nn.Sequential()
    if stride != 1 or in_planes != self.expansion*planes:
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False),
            nn.BatchNorm2d(self.expansion*planes)
        )

  def forward(self, x):
    out = F.relu(self.bn1(self.conv1(x)))
    out = self.bn2(self.conv2(out))
    out += self.shortcut(x)
    out = F.relu(out)
    return out


class Bottleneck(nn.Module):
  expansion = 4

  def __init__(self, in_planes, planes, stride=1):
    super(Bottleneck, self).__init__()
    self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
    self.bn1 = nn.BatchNorm2d(planes)
    self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
    self.bn2 = nn.BatchNorm2d(planes)
    self.conv3 = nn.Conv2d(planes, self.expansion*planes, kernel_size=1, bias=False)
    self.bn3 = nn.BatchNorm2d(self.expansion*planes)

    self.shortcut = nn.Sequential()
    if stride != 1 or in_planes != self.expansion*planes:
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False),
            nn.BatchNorm2d(self.expansion*planes)
        )

  def forward(self, x):
    out = F.relu(self.bn1(self.conv1(x)))
    out = F.relu(self.bn2(self.conv2(out)))
    out = self.bn3(self.conv3(out))
    out += self.shortcut(x)
    out = F.relu(out)
    return out


class ResNet(nn.Module):
  def __init__(self, block, num_blocks, num_classes=100):
    super(ResNet, self).__init__()
    self.in_planes = 64

    self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    self.bn1 = nn.BatchNorm2d(64)
    self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
    self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
    self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
    self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
    self.linear = nn.Linear(512*block.expansion, num_classes)

  def _make_layer(self, block, planes, num_blocks, stride):
    strides = [stride] + [1]*(num_blocks-1)
    layers = []
    for stride in strides:
      layers.append(block(self.in_planes, planes, stride))
      self.in_planes = planes * block.expansion
    return nn.Sequential(*layers)

  def forward(self, x):
    out = F.relu(self.bn1(self.conv1(x)))
    out = self.layer1(out)
    out = self.layer2(out)
    out = self.layer3(out)
    out = self.layer4(out)
    out = F.avg_pool2d(out, 4)
    out = out.view(out.size(0), -1)
    out = self.linear(out)
    return out


def ResNet18():
  return ResNet(BasicBlock, [2, 2, 2, 2])


def ResNet34():
  return ResNet(BasicBlock, [3, 4, 6, 3])


def ResNet50():
  return ResNet(Bottleneck, [3, 4, 6, 3])

##CIFAR-10-en és CIFAR-100-on betanított modell betöltése

In [ ]:
print(torch.hub.list("chenyaofo/pytorch-cifar-models", force_reload=True))

/usr/local/lib/python3.10/dist-packages/torch/hub.py:293: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or list(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use list(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/zipball/master" to /root/.cache/torch/hub/master.zip


['cifar100_mobilenetv2_x0_5', 'cifar100_mobilenetv2_x0_75', 'cifar100_mobilenetv2_x1_0', 'cifar100_mobilenetv2_x1_4', 'cifar100_repvgg_a0', 'cifar100_repvgg_a1', 'cifar100_repvgg_a2', 'cifar100_resnet20', 'cifar100_resnet32', 'cifar100_resnet44', 'cifar100_resnet56', 'cifar100_shufflenetv2_x0_5', 'cifar100_shufflenetv2_x1_0', 'cifar100_shufflenetv2_x1_5', 'cifar100_shufflenetv2_x2_0', 'cifar100_vgg11_bn', 'cifar100_vgg13_bn', 'cifar100_vgg16_bn', 'cifar100_vgg19_bn', 'cifar100_vit_b16', 'cifar100_vit_b32', 'cifar100_vit_h14', 'cifar100_vit_l16', 'cifar100_vit_l32', 'cifar10_mobilenetv2_x0_5', 'cifar10_mobilenetv2_x0_75', 'cifar10_mobilenetv2_x1_0', 'cifar10_mobilenetv2_x1_4', 'cifar10_repvgg_a0', 'cifar10_repvgg_a1', 'cifar10_repvgg_a2', 'cifar10_resnet20', 'cifar10_resnet32', 'cifar10_resnet44', 'cifar10_resnet56', 'cifar10_shufflenetv2_x0_5', 'cifar10_shufflenetv2_x1_0', 'cifar10_shufflenetv2_x1_5', 'cifar10_shufflenetv2_x2_0', 'cifar10_vgg11_bn', 'cifar10_vgg13_bn', 'cifar10_vgg16_b

In [ ]:
cifar10_model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=True)
cifar100_model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar100_resnet20", pretrained=True)

Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/resnet/cifar10_resnet20-4118986f.pt" to /root/.cache/torch/hub/checkpoints/cifar10_resnet20-4118986f.pt
100%|██████████| 1.09M/1.09M [00:00<00:00, 18.4MB/s]
Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/resnet/cifar100_resnet20-23dac2f1.pt" to /root/.cache/torch/hub/checkpoints/cifar100_resnet20-23dac2f1.pt
100%|██████████| 1.11M/1.11M [00:00<00:00, 20.8MB/s]


In [ ]:
cifar10_model

CifarResNet(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias

In [ ]:
cifar100_model

CifarResNet(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias

# **1\. Feladat - Transzfer tanítás előtt (x pont)**

Töltsd be két ResNet-18/20 modellt: az egyik legyen a CIFAR-100 adathalmaz osztályozására betanítva, a másik pedig a CIFAR-10 osztályozására. Írd le, hogy milyen pontosságot érnek el a betöltött modellek a megfelelő teszt-halmazokon. Cseréld ki a CIFAR-100 adaton tanított modell utolsó rétegét a CIFAR-10 adaton tanított modellével. Számold ki az így kapott háló pontosságát a CIFAR-10 tanító és teszt adathalmazán külön-külön, és hasonlítsd össze ezeket egy véletlenszerűen inicializált háló (amelynek a kimeneti rétegét szintén lecseréled a CIFAR-10-en tanított modellével) teljesítményével (tehát összesen 4 számot kell megadni), hogy kiderüljön, mennyire történik meg a tudás-transzfer továbbtanítás nélkül a CIFAR-100-ról.  

Tipp: célszerű külön függvényt írni adott modell adott halmazon való kiértékelésére.

# **2\. Feladat - Transzfer tanítás (x pont)**

Csakis az utolsó kimeneti réteg lecserélésével (megfelelő kimeneti neuront tartalmazó, de véletlenszerű paraméterekkel inicializált rétegre kell cserélni), és a többi réteg befagyasztásával tanítsd tovább a CIFAR-10 tanító halmazán a CIFAR-100-on betanított hálót, alkalmasan megválasztott paraméterekkel. Ábrázold egy ábrán a továbbtanítási epochok függvényében, hogyan alakul a háló pontossága a CIFAR-10, illetve a CIFAR-100 teszt-halmazán.


# **3\. Feladat - Modellek távolsága (x pont)**

Számold ki 3 modell paramétereinek a páronkénti $L_2$-távolságát: a három összehasonlított modell legyen a véletlenszerűen inicializiált, az 1. feladatban CIFAR-100-on tanított modell a lecserélt kiemneti réteggel és a 2. feladatban a CIFAR-10-en továbbtanított modell. Jelenítsd meg ezeket egy 3x3-as mátrixban, ami szimmetrikus, és egy modell távolsága önmagával 0.

# **4\. Feladat - Több réteg továbbtanítása (x pont)**
Csak az utolsó réteg tanítása helyett itt már megengedjük, hogy a háló alsóbb rétegei is tanuljanak: a bemenethez közeli rétegeket  az második reziduális blokkal bezárólag is továbbtanítva, lehet-e jobb pontosságot elérni a CIFAR-10 tesz-halmazán? Mi van akkor, ha a csak a második reziduális blokk utáni rétegek tanítása megengedett?

# **5. Feladat - Kevés adaton való tanítás (x pont)**

Ha a CIFAR-10 tanítóadatainak csak az 50%-ához férsz hozzá, el tudsz-e érni ugyanolyan jó pontosságot a CIFAR-100-en tanított modell transzfertanításával ezen a kevesebb adaton, mint a teljes CIFAR-10-en tanított modell? Ha sikerül, írd le milyen beállításokkal. Az adatok 20%-ával is sikerül ugyanez?